# CatBoost v4: Baseline-First

## Стратегия: воспроизвести baseline, затем точечно улучшить

### Шаг A: обучение с параметрами baseline (iterations=2000, lr=0.03, depth=6)
### Шаг B: Optuna тюнинг (15 триалов, GPU)
### Шаг C: финальное обучение с лучшими параметрами

### Ключевые отличия от v3:
- **Single-fold training** (как baseline): train fold N-1 → val fold N
- **stride=14** (как baseline, через `8_data_v4.ipynb`)
- **Без feature selection** — модель сама отбирает
- Параметры стартуют с baseline-значений

In [5]:
import polars as pl
import pandas as pd
import numpy as np
import json
from pathlib import Path
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import optuna

FEATURES_DIR = Path("../data/processed/features_v4")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 2

def rmsle_score(y_true, y_pred):
    y_pred_clipped = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred_clipped)))

def gini_normalized(y_true, y_pred):
    def _gini(actual, predicted):
        n = len(actual)
        indices = np.argsort(-predicted)
        sorted_actual = actual[indices]
        cumulative = np.cumsum(sorted_actual)
        gini_sum = cumulative.sum() / (sorted_actual.sum() + 1e-9) - (n + 1) / 2
        return gini_sum / n
    return _gini(y_true, y_pred) / (_gini(y_true, y_true) + 1e-9)

def rmspe_total_gmv(y_true, y_pred):
    total_true = y_true.sum()
    total_pred = max(y_pred.sum(), 1e-8)
    return abs(total_true - total_pred) / total_true * 100

def load_fold(fold_path: Path) -> pd.DataFrame:
    return pl.read_parquet(fold_path / "batch_*.parquet").to_pandas()

In [6]:
print("Загрузка данных...")
all_folds = []
for fold_idx in range(N_FOLDS):
    fold_df = load_fold(FEATURES_DIR / f"fold_{fold_idx:02d}")
    all_folds.append(fold_df)
    n_buyers = (fold_df["target"] > 0).sum()
    print(f"  fold_{fold_idx:02d}: {len(fold_df):,} | buyers={n_buyers:,} ({n_buyers/len(fold_df)*100:.1f}%)")

test_df = load_fold(FEATURES_DIR / "fold_test")
print(f"  fold_test: {len(test_df):,}")

drop_cols = ["user_id", "anchor_date", "target"]
features = [c for c in test_df.columns if c not in drop_cols]
print(f"\nФичей: {len(features)}")

Загрузка данных...
  fold_00: 250,000 | buyers=133,886 (53.6%)
  fold_01: 250,000 | buyers=135,165 (54.1%)
  fold_test: 250,000

Фичей: 392


## Шаг A: Baseline параметры (single-fold training)

Точная копия baseline CV: train fold N-1 → val fold N.

In [ ]:
baseline_params = {
    'iterations': 2000,
    'learning_rate': 0.03,
    'depth': 6,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'task_type': 'GPU',
    'devices': '0',
    'random_seed': 42,
    'od_type': 'Iter',
    'early_stopping_rounds': 150,
    'verbose': 250,
}

print("=== Шаг A: Baseline-style single-fold training ===")
test_predictions_a = np.zeros(len(test_df))
fold_scores_a = []
fold_gini_a = []
fold_rmspe_a = []

for val_idx in range(1, N_FOLDS):
    train_df = all_folds[val_idx - 1]
    val_df   = all_folds[val_idx]
    
    print(f"\n--- Train: fold_{val_idx-1:02d} ({len(train_df):,}) | Val: fold_{val_idx:02d} ({len(val_df):,}) ---")
    
    X_train = train_df[features]
    y_train_log = np.log1p(np.clip(train_df["target"], 0, None))
    X_val = val_df[features]
    y_val = val_df["target"].values
    y_val_log = np.log1p(np.clip(y_val, 0, None))
    
    model = CatBoostRegressor(**baseline_params)
    model.fit(Pool(X_train, y_train_log), eval_set=Pool(X_val, y_val_log))
    
    val_pred = np.expm1(np.clip(model.predict(X_val), 0, None))
    
    score = rmsle_score(y_val, val_pred)
    gini  = gini_normalized(y_val, val_pred)
    rmspe = rmspe_total_gmv(y_val, val_pred)
    
    fold_scores_a.append(score)
    fold_gini_a.append(gini)
    fold_rmspe_a.append(rmspe)
    
    print(f"  RMSLE: {score:.5f} | Gini: {gini:.4f} | RMSPE: {rmspe:.2f}%")
    
    test_pred_log = model.predict(test_df[features])
    test_predictions_a += np.expm1(np.clip(test_pred_log, 0, None)) / (N_FOLDS - 1)

print(f"\n=== Шаг A итоги ===")
print(f"Mean RMSLE: {np.mean(fold_scores_a):.5f} ± {np.std(fold_scores_a):.5f}")
print(f"Mean Gini:  {np.mean(fold_gini_a):.4f}")
print(f"Mean RMSPE: {np.mean(fold_rmspe_a):.2f}%")

submit_a = test_df[["user_id"]].copy()
submit_a["predict"] = np.clip(test_predictions_a, 0, None)
submit_a.to_csv(Path("../data/processed/catboost_v4a_baseline_submission.csv"), index=False)
print(f"\nСохранено: catboost_v4a_baseline_submission.csv")
print(f"  mean={submit_a['predict'].mean():.2f} | median={submit_a['predict'].median():.2f}")

=== Шаг A: Baseline-style single-fold training ===

--- Train: fold_00 (250,000) | Val: fold_01 (250,000) ---
0:	learn: 2.2638575	test: 2.2596438	best: 2.2596438 (0)	total: 117ms	remaining: 3m 53s
250:	learn: 1.6856361	test: 1.6723533	best: 1.6723533 (250)	total: 3.06s	remaining: 21.4s
500:	learn: 1.6774814	test: 1.6693647	best: 1.6693647 (500)	total: 5.91s	remaining: 17.7s
750:	learn: 1.6707225	test: 1.6676765	best: 1.6676765 (750)	total: 8.61s	remaining: 14.3s
1000:	learn: 1.6646071	test: 1.6665827	best: 1.6665827 (1000)	total: 11.2s	remaining: 11.2s
1250:	learn: 1.6588022	test: 1.6656701	best: 1.6656701 (1250)	total: 13.9s	remaining: 8.29s
1500:	learn: 1.6532232	test: 1.6647401	best: 1.6647401 (1500)	total: 16.5s	remaining: 5.5s
1750:	learn: 1.6477463	test: 1.6639980	best: 1.6639980 (1750)	total: 19.2s	remaining: 2.73s
1999:	learn: 1.6424919	test: 1.6633230	best: 1.6633230 (1999)	total: 21.8s	remaining: 0us
bestTest = 1.663322954
bestIteration = 1999
  RMSLE: 1.66332 | Gini: 0.7397 

## Шаг B: Optuna (GPU, single-fold CV)

Тюним на тех же single-fold splits.

In [ ]:
N_OPTUNA_TRIALS = 20

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 1500, 3500, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'depth': trial.suggest_int('depth', 5, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.01, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'task_type': 'GPU',
        'devices': '0',
        'random_seed': 42,
        'od_type': 'Iter',
        'early_stopping_rounds': 100,
        'verbose': False,
    }
    
    scores = []
    for val_idx in range(1, N_FOLDS):
        train_df = all_folds[val_idx - 1]
        val_df   = all_folds[val_idx]
        
        y_train_log = np.log1p(np.clip(train_df["target"], 0, None))
        y_val = val_df["target"].values
        y_val_log = np.log1p(np.clip(y_val, 0, None))
        
        model = CatBoostRegressor(**params)
        model.fit(Pool(train_df[features], y_train_log), eval_set=Pool(val_df[features], y_val_log))
        
        val_pred = np.expm1(np.clip(model.predict(val_df[features]), 0, None))
        scores.append(rmsle_score(y_val, val_pred))
        
        trial.report(np.mean(scores), val_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return np.mean(scores)

def print_callback(study, trial):
    status = "PRUNED" if trial.state == optuna.trial.TrialState.PRUNED else f"RMSLE={trial.value:.5f}"
    print(f"  Trial {trial.number+1:2d}/{N_OPTUNA_TRIALS}: {status} "
          f"(best: {study.best_value:.5f}, trial #{study.best_trial.number+1})")

print(f"=== Шаг B: Optuna ({N_OPTUNA_TRIALS} триалов, GPU, single-fold) ===")
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[print_callback])

print(f"\nЛучший RMSLE: {study.best_value:.5f}")
print("Параметры:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-08-17 04:48:52,544] A new study created in memory with name: no-name-52039368-43f1-4bd5-bacc-537dfa2283bb


=== Шаг B: Optuna (20 триалов, GPU, single-fold) ===
  Trial  1/20: RMSLE=1.65568 (best: 1.65568, trial #1)
  Trial  2/20: RMSLE=1.66001 (best: 1.65568, trial #1)
  Trial  3/20: RMSLE=1.66467 (best: 1.65568, trial #1)
  Trial  4/20: RMSLE=1.66270 (best: 1.65568, trial #1)
  Trial  5/20: RMSLE=1.66234 (best: 1.65568, trial #1)
  Trial  6/20: RMSLE=1.65775 (best: 1.65568, trial #1)
  Trial  7/20: RMSLE=1.65688 (best: 1.65568, trial #1)
  Trial  8/20: PRUNED (best: 1.65568, trial #1)
  Trial  9/20: PRUNED (best: 1.65568, trial #1)
  Trial 10/20: PRUNED (best: 1.65568, trial #1)
  Trial 11/20: RMSLE=1.64964 (best: 1.64964, trial #11)
  Trial 12/20: RMSLE=1.65024 (best: 1.64964, trial #11)
  Trial 13/20: RMSLE=1.64927 (best: 1.64927, trial #13)
  Trial 14/20: RMSLE=1.65237 (best: 1.64927, trial #13)
  Trial 15/20: RMSLE=1.64919 (best: 1.64919, trial #15)
  Trial 16/20: RMSLE=1.65427 (best: 1.64919, trial #15)
  Trial 17/20: PRUNED (best: 1.64919, trial #15)
  Trial 18/20: PRUNED (best: 1.64

## Шаг C: Финальное обучение

Два варианта:
1. **Single-fold** (как baseline)
2. **Cumulative** (train 0..N-1 → val N) — для сравнения

In [ ]:
best_params = study.best_params

print("=== Шаг C: Single-fold с Optuna параметрами ===")
test_predictions_single = np.zeros(len(test_df))
fold_scores_single = []
models_single = []

for val_idx in range(1, N_FOLDS):
    train_df = all_folds[val_idx - 1]
    val_df   = all_folds[val_idx]
    
    print(f"\n--- Train: fold_{val_idx-1:02d} ({len(train_df):,}) | Val: fold_{val_idx:02d} ({len(val_df):,}) ---")
    
    y_train_log = np.log1p(np.clip(train_df["target"], 0, None))
    y_val = val_df["target"].values
    y_val_log = np.log1p(np.clip(y_val, 0, None))
    
    model = CatBoostRegressor(
        **best_params,
        loss_function='RMSE', eval_metric='RMSE',
        task_type='GPU', devices='0', random_seed=42,
        early_stopping_rounds=150, verbose=250
    )
    model.fit(Pool(train_df[features], y_train_log), eval_set=Pool(val_df[features], y_val_log))
    models_single.append(model)
    
    val_pred = np.expm1(np.clip(model.predict(val_df[features]), 0, None))
    score = rmsle_score(y_val, val_pred)
    gini  = gini_normalized(y_val, val_pred)
    fold_scores_single.append(score)
    print(f"  RMSLE: {score:.5f} | Gini: {gini:.4f}")
    
    test_predictions_single += np.expm1(np.clip(model.predict(test_df[features]), 0, None)) / (N_FOLDS - 1)
    model.save_model(str(MODELS_DIR / f"catboost_v4_single_fold_{val_idx}.cbm"))

print(f"\nSingle-fold RMSLE: {np.mean(fold_scores_single):.5f} ± {np.std(fold_scores_single):.5f}")

print("\n=== Шаг C: Cumulative с Optuna параметрами ===")
test_predictions_cumul = np.zeros(len(test_df))
fold_scores_cumul = []

for val_idx in range(1, N_FOLDS):
    train_dfs = [all_folds[i] for i in range(val_idx)]
    train_df = pd.concat(train_dfs, ignore_index=True)
    val_df   = all_folds[val_idx]
    
    train_str = "+".join(str(i) for i in range(val_idx))
    print(f"\n--- Train: [{train_str}] ({len(train_df):,}) | Val: fold_{val_idx:02d} ({len(val_df):,}) ---")
    
    y_train_log = np.log1p(np.clip(train_df["target"], 0, None))
    y_val = val_df["target"].values
    y_val_log = np.log1p(np.clip(y_val, 0, None))
    
    model = CatBoostRegressor(
        **best_params,
        loss_function='RMSE', eval_metric='RMSE',
        task_type='GPU', devices='0', random_seed=42,
        early_stopping_rounds=150, verbose=250
    )
    model.fit(Pool(train_df[features], y_train_log), eval_set=Pool(val_df[features], y_val_log))
    
    val_pred = np.expm1(np.clip(model.predict(val_df[features]), 0, None))
    score = rmsle_score(y_val, val_pred)
    gini  = gini_normalized(y_val, val_pred)
    fold_scores_cumul.append(score)
    print(f"  RMSLE: {score:.5f} | Gini: {gini:.4f}")
    
    test_predictions_cumul += np.expm1(np.clip(model.predict(test_df[features]), 0, None)) / (N_FOLDS - 1)
    model.save_model(str(MODELS_DIR / f"catboost_v4_cumul_fold_{val_idx}.cbm"))

print(f"\nCumulative RMSLE: {np.mean(fold_scores_cumul):.5f} ± {np.std(fold_scores_cumul):.5f}")

print(f"\n{'='*50}")
print(f"Single-fold: {np.mean(fold_scores_single):.5f}")
print(f"Cumulative:  {np.mean(fold_scores_cumul):.5f}")
better = "single" if np.mean(fold_scores_single) < np.mean(fold_scores_cumul) else "cumulative"
print(f"Лучше: {better}")

=== Шаг C: Single-fold с Optuna параметрами ===

--- Train: fold_00 (250,000) | Val: fold_01 (250,000) ---
0:	learn: 2.2425250	test: 2.2382862	best: 2.2382862 (0)	total: 27.9ms	remaining: 1m 23s
250:	learn: 1.6631835	test: 1.6683849	best: 1.6683849 (250)	total: 4.87s	remaining: 53.3s
500:	learn: 1.6327487	test: 1.6643156	best: 1.6643156 (500)	total: 9.57s	remaining: 47.7s
750:	learn: 1.6059101	test: 1.6617875	best: 1.6617875 (750)	total: 14.3s	remaining: 42.9s
1000:	learn: 1.5801469	test: 1.6596203	best: 1.6596036 (996)	total: 19.1s	remaining: 38.2s
1250:	learn: 1.5554078	test: 1.6577466	best: 1.6577193 (1242)	total: 24s	remaining: 33.5s
1500:	learn: 1.5317528	test: 1.6564217	best: 1.6564199 (1499)	total: 28.8s	remaining: 28.8s
1750:	learn: 1.5091114	test: 1.6553186	best: 1.6553180 (1745)	total: 33.6s	remaining: 24s
2000:	learn: 1.4867983	test: 1.6536549	best: 1.6536511 (1995)	total: 38.5s	remaining: 19.2s
2250:	learn: 1.4657455	test: 1.6525204	best: 1.6525204 (2250)	total: 43.4s	remai

## Сабмиты и бленды

In [ ]:
PROCESSED = Path("../data/processed")

for name, preds in [("single", test_predictions_single), ("cumul", test_predictions_cumul)]:
    sub = test_df[["user_id"]].copy()
    sub["predict"] = np.clip(preds, 0, None)
    fname = f"catboost_v4_{name}_submission.csv"
    sub.to_csv(PROCESSED / fname, index=False)
    print(f"{fname}: mean={sub['predict'].mean():.2f}, median={sub['predict'].median():.2f}")

blend_sc = 0.5 * test_predictions_single + 0.5 * test_predictions_cumul
sub_sc = test_df[["user_id"]].copy()
sub_sc["predict"] = np.clip(blend_sc, 0, None)
sub_sc.to_csv(PROCESSED / "catboost_v4_blend_single_cumul.csv", index=False)
print(f"blend_single_cumul: mean={sub_sc['predict'].mean():.2f}")

try:
    bl = pd.read_csv(PROCESSED / "catboost_baseline_submission.csv").sort_values("user_id").reset_index(drop=True)
    v4_sorted = test_df[["user_id"]].copy()
    v4_sorted["predict"] = np.clip(test_predictions_single, 0, None)
    v4_sorted = v4_sorted.sort_values("user_id").reset_index(drop=True)
    
    for alpha in [0.3, 0.5, 0.7]:
        blend = alpha * v4_sorted["predict"].values + (1 - alpha) * bl["predict"].values
        fname = f"blend_v4_{int(alpha*100)}_bl{int((1-alpha)*100)}.csv"
        pd.DataFrame({"user_id": bl["user_id"], "predict": np.clip(blend, 0, None)}).to_csv(
            PROCESSED / fname, index=False)
        print(f"{fname}: mean={blend.mean():.2f}")
except Exception as e:
    print(f"baseline CSV не найден: {e}")

importances = models_single[-1].get_feature_importance()
imp_dict = dict(zip(features, importances))
sorted_imp = sorted(imp_dict.items(), key=lambda x: x[1], reverse=True)

print(f"\nТоп-20 фичей:")
for feat, imp in sorted_imp[:20]:
    print(f"  {feat}: {imp:.2f}%")

v4_imp = {"features_ranked": [{"name": f, "importance": round(imp, 4)} for f, imp in sorted_imp]}
with open(PROCESSED / "feature_importance_v4.json", "w") as f:
    json.dump(v4_imp, f, indent=2)
print("Feature importance v4 сохранено.")

catboost_v4_single_submission.csv: mean=36.28, median=6.20
catboost_v4_cumul_submission.csv: mean=36.29, median=6.22
blend_single_cumul: mean=36.29
blend_v4_30_bl70.csv: mean=39.06
blend_v4_50_bl50.csv: mean=38.27
blend_v4_70_bl30.csv: mean=37.47

Топ-20 фичей:
  recency_ord_days: 1.33%
  freq_ord_365d: 1.33%
  user_age_days: 1.29%
  to_ord_sum_365d: 1.22%
  freq_ord_270d: 1.21%
  freq_ord_90d: 1.18%
  trend_gmv_90d_365d: 1.11%
  gmv_search_sum_365d: 0.96%
  freq_ord_180d: 0.92%
  recency_cat_days: 0.88%
  trend_search_7d_30d: 0.82%
  search_mean_365d: 0.82%
  aov_90d: 0.79%
  recency_search_days: 0.78%
  aov_180d: 0.75%
  trend_gmv_30d_90d: 0.74%
  cat_mean_365d: 0.72%
  to_ord_mean_365d: 0.69%
  trend_gmv_30d_180d: 0.67%
  search_to_ord_sum_270d: 0.67%
Feature importance v4 сохранено.
